# TD4: Identifier les personnes et le contenu d'un message à envoyer d'une requête à un assistant virtuel

Ici, on va entraîner un modèle NER, en utilisant du transfer learning. <br/>
Notre dataset est petit, il faut surtout éviter l'overfitting.<br/>

Je vais prendre un modèle pré-entraîner et graduellement unfreeze des layers jusqu'au moment où le modèle overfit <br/>
Au passage, dans une layer d'attention, les parties "query, key, value" (qui disent où l'attention doit se porter) sont celles qui donnent toute cette souplesse à cette architecture -> ce sont les plus rapides à overfit. <br/>
Je vais d'abord apprendre les "feed-forward" layers. <br/>
Quand elles seront apprises (avant et après la partie query, key, value), elles donneront des résultats cohérents avec le signal à prédire (si le token est une personne, du message, ou rien). <br/>
Quand les parties (query, key, value) ont des feed-forward layers cohérentes (avant et après), on pourra les ré-apprendre. <br/>

En pratique, j'ai pu apprendre les 2 dernières layers d'attention avant que le modèle n'overfit (voir le notebook)

In [1]:
import csv
import json
import matplotlib.pyplot as plt
import pandas as pd
import torch
import transformers

In [2]:
model_name = "distilbert-base-cased"

# Load Presto data

In [3]:
df = pd.read_csv("../data/raw/presto/train_2.csv")
df["words"] = df["words"].apply(json.loads)
df["labels"] = df["labels"].apply(json.loads)

In [4]:
df_dev = pd.read_csv("../data/raw/presto/dev.csv")
df_dev["words"] = df_dev["words"].apply(json.loads)
df_dev["labels"] = df_dev["labels"].apply(json.loads)

In [28]:
df_test = pd.read_csv("../data/raw/presto/test.csv")
df_test["words"] = df_test["words"].apply(json.loads)
df_test["labels"] = df_test["labels"].apply(json.loads)

In [29]:
sentences_train = list(df["words"])
sentences_dev = list(df_dev["words"])
sentences_test = list(df_test["words"])

labels_train = list(df["labels"])
labels_dev = list(df_dev["labels"])
labels_test = list(df_test["labels"])

In [6]:
all_labels = set(lab for labels in labels_train for lab in labels)

label_to_int = {lab: i for i, lab in enumerate(all_labels)}

def translate_label_to_id(labels):
    return [label_to_int[lab] for lab in labels]

In [7]:
label_to_int

{0: 0, 'person': 1, 'content': 2}

In [30]:
labels_train = [translate_label_to_id(labels) for labels in labels_train]
labels_dev = [translate_label_to_id(labels) for labels in labels_dev]
labels_test = [translate_label_to_id(labels) for labels in labels_test]

# Tokenize data

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True)

In [11]:
def tokenize_and_align_labels(sentences, ner_tags):
    tokenized_inputs = tokenizer(
        sentences,
        truncation=True,
        is_split_into_words=True,
    )
    labels = []
    for i, label in enumerate(ner_tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [12]:
tokenized_train = tokenize_and_align_labels(sentences_train, labels_train)

In [13]:
tokenized_dev = tokenize_and_align_labels(sentences_dev, labels_dev)

In [31]:
tokenized_test = tokenize_and_align_labels(sentences_test, labels_test)

In [32]:
from datasets import Dataset

dataset_train = Dataset.from_dict(tokenized_train)
dataset_dev = Dataset.from_dict(tokenized_dev)
dataset_test = Dataset.from_dict(tokenized_test)

In [33]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Evaluation functions

In [16]:
import numpy as np
import evaluate

seqeval = evaluate.load("seqeval")

labels = list(label_to_int.values())
label_list = [str(lab) for lab in label_to_int]

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Load model

In [17]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=len(labels)
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Freeze all but last layer

In [18]:
for name, param in model.base_model.named_parameters():
  param.requires_grad = False

for name, param in model.base_model.named_parameters():
    if (
        any(layer_name in name for layer_name in ["layer.5.output", "layer.5.ffn.lin"])
    ):
        param.requires_grad = True

In [19]:
training_args = TrainingArguments(
    output_dir="freeze_all_but_last_layer",
    learning_rate=2e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.454683,0.000000,0.000000,0.000000,0.942828
2,No log,0.291080,0.000000,0.000000,0.000000,0.942828
3,No log,0.264833,0.000000,0.000000,0.000000,0.942828
4,No log,0.259257,0.000000,0.000000,0.000000,0.942828
5,No log,0.257735,0.000000,0.000000,0.000000,0.942828


/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: content seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: person seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/sit

TrainOutput(global_step=115, training_loss=0.4213357676630435, metrics={'train_runtime': 181.3795, 'train_samples_per_second': 80.797, 'train_steps_per_second': 0.634, 'total_flos': 102742913312550.0, 'train_loss': 0.4213357676630435, 'epoch': 5.0})

## Freeze all but 2 last layers

In [20]:
for name, param in model.base_model.named_parameters():
  param.requires_grad = False

for name, param in model.base_model.named_parameters():
    if (
        any(layer_name in name for layer_name in ["layer.5.output", "layer.5.ffn.lin", "layer.4.ffn.lin", "layer.4.output"])
    ):
        param.requires_grad = True

In [21]:
training_args = TrainingArguments(
    output_dir="freeze_all_but_2_last_layers",
    learning_rate=2e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.191346,0.222222,0.008264,0.015936,0.943211
2,No log,0.141744,0.442080,0.386364,0.412348,0.958780
3,No log,0.125715,0.480176,0.450413,0.464819,0.961205
4,No log,0.119541,0.501057,0.489669,0.495298,0.964267
5,No log,0.117375,0.513978,0.493802,0.503688,0.965288


/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: content seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: person seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: content seems not to be NE tag.
  warnings.warn('{

TrainOutput(global_step=115, training_loss=0.1749589505402938, metrics={'train_runtime': 213.334, 'train_samples_per_second': 68.695, 'train_steps_per_second': 0.539, 'total_flos': 102742913312550.0, 'train_loss': 0.1749589505402938, 'epoch': 5.0})

## Release last attention layer

In [22]:
for name, param in model.base_model.named_parameters():
  param.requires_grad = False

for name, param in model.base_model.named_parameters():
    if (
        any(layer_name in name for layer_name in ["layer.5", "layer.4.ffn.lin", "layer.4.output"])
    ):
        param.requires_grad = True

In [23]:
training_args = TrainingArguments(
    output_dir="unfreeze_last_attention",
    learning_rate=2e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.105060,0.539823,0.504132,0.521368,0.966437
2,No log,0.095890,0.560000,0.607438,0.582755,0.972435
3,No log,0.090147,0.565134,0.609504,0.586481,0.972945
4,No log,0.087553,0.592105,0.650826,0.620079,0.974349
5,No log,0.086544,0.592105,0.650826,0.620079,0.974477


/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: content seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: person seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: content seems not to be NE tag.
  warnings.warn('{

TrainOutput(global_step=115, training_loss=0.0908198066379713, metrics={'train_runtime': 212.1529, 'train_samples_per_second': 69.078, 'train_steps_per_second': 0.542, 'total_flos': 102742913312550.0, 'train_loss': 0.0908198066379713, 'epoch': 5.0})

## Release last-3 feed forward layer

In [24]:
for name, param in model.base_model.named_parameters():
  param.requires_grad = False

for name, param in model.base_model.named_parameters():
    if (
        any(layer_name in name for layer_name in ["layer.5", "layer.4.ffn.lin", "layer.4.output", "layer.3.ffn.lin", "layer.3.output"])
    ):
        param.requires_grad = True

In [25]:
training_args = TrainingArguments(
    output_dir="unfreeze_3_last_layers",
    learning_rate=1e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.081404,0.616601,0.644628,0.630303,0.975753
2,No log,0.077833,0.621974,0.690083,0.654261,0.977157
3,No log,0.075342,0.651601,0.714876,0.681773,0.978688
4,No log,0.074175,0.643911,0.721074,0.680312,0.978688
5,No log,0.073509,0.659696,0.716942,0.687129,0.979071


/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: content seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: person seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: content seems not to be NE tag.
  warnings.warn('{

TrainOutput(global_step=115, training_loss=0.06183423166689665, metrics={'train_runtime': 245.3167, 'train_samples_per_second': 59.739, 'train_steps_per_second': 0.469, 'total_flos': 102742913312550.0, 'train_loss': 0.06183423166689665, 'epoch': 5.0})

# Evaluate model on test

In [39]:
from tqdm import tqdm

In [67]:
preds = []
labels = []

for row in tqdm(dataset_test):
    pred = model(torch.tensor([row["input_ids"]]))
    preds += list(np.argmax(pred.logits.detach().numpy(), axis=2)[0])
    labels += row["labels"]


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1991/1991 [00:35<00:00, 55.50it/s]


In [68]:
ids_ok = {i for i, label in enumerate(labels) if label != -100}
preds = [pred for i, pred in enumerate(preds) if i in ids_ok]
labels = [lab for i, lab in enumerate(labels) if i in ids_ok]

In [69]:
from sklearn.metrics import accuracy_score

accuracy_score(labels, preds)

0.975748389541493

In [61]:
preds

[]